In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import pandas as pd
from pandas.core.series import Series
from PIL import Image

import plotly.graph_objects as go


dataset_classes = sorted(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 
			'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 
			'del', 'nothing', 'space'])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_path = "../Data/asl_alphabet_train/asl_alphabet_train"
test_path = "../Data/asl_alphabet_test/asl_alphabet_test/"

IMAGE_SIZE = 64
BATCH_SIZE = 64

train_path_df = pd.read_csv("../Data/train_path_datafram.csv")
device

device(type='cuda')

In [2]:
# Training transforms with Augmentation (Bonus)
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(15),           # Randomly rotate the image by up to 15 degrees
    transforms.ColorJitter(brightness=0.2),  # Randomly change brightness
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), # Randomly shift the image
    transforms.ToTensor(),
])

# Validation/Test transforms (Clean data)
test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

In [3]:
X_train, X_test, Y_train, Y_test = train_test_split(train_path_df['image_path'], train_path_df['label'], test_size = 0.2, random_state=54)
X_train, X_valid, Y_train, Y_valid = train_test_split(X_train,  Y_train, test_size = 0.25, random_state=54)


In [4]:
class ASLDataset(Dataset):
	def __init__(self, X: Series, Y: Series, transform=None):
		self.X = X.reset_index(drop = True)
		self.Y = Y.reset_index(drop = True)
		# self.dataframe = dataframe.reset_index(drop=True)
		self.transform = transform
		# Create a mapping from text labels (A, B, del...) to integers (0, 1, 2...)
		self.labels = sorted(self.Y.unique())
		self.idx = {label: idx for idx, label in enumerate(self.labels)}
	
	def __len__(self):
		return len(self.X)
	
	def __getitem__(self, idx):
		# Fetch image path and class string
		img_path = self.X[idx]
		class_name = self.Y[idx]
	       	
		       # Load the image lazily (only when requested)
		image = Image.open(img_path).convert('RGB')
		label_idx = self.idx[class_name]
		
		# Apply the resizing and normalization transforms
		if self.transform:
			image = self.transform(image)
					
		return image, torch.tensor(label_idx, dtype=torch.long)

# 3. Instantiate the Train, Validation, and Test Datasets
train_dataset = ASLDataset(X_train, Y_train, transform=train_transforms)
valid_dataset = ASLDataset(X_valid, Y_valid, transform=test_transforms)
test_dataset = ASLDataset(X_test, Y_test, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 9)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 3)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 3)

In [5]:
class CNN(nn.Module):
        def __init__(self, num_classes):
                super().__init__()
                self.POOL_SIZE = 2
                self.L1_OUT_CHANNEL  = 32
                self.L1_KERNEL_SIZE  = 3
                self.L2_OUT_CHANNEL  = 64
                self.L2_KERNEL_SIZE  = 3
                self.L3_OUT_CHANNEL  = 128
                self.L3_KERNEL_SIZE  = 3
                self.DROPOUT_P = 0.5
                self.L1_NEURON = 512
                self.FINAL_DIM = IMAGE_SIZE // 8 # x Layers --> 2 ** x
                
                
                self.pool = nn.MaxPool2d(
                        kernel_size = self.POOL_SIZE,
                        stride = self.POOL_SIZE
                )
                self.relu = nn.ReLU()
                
                # Layer 1
                
                self.conv1 = nn.Conv2d(
                        in_channels = 3,
                        out_channels = self.L1_OUT_CHANNEL,
                        kernel_size =self. L1_KERNEL_SIZE,
                        padding = (self.L1_KERNEL_SIZE - 1) // 2
                )
                self.batch_norm1 = nn.BatchNorm2d(self.L1_OUT_CHANNEL)
                
                # Layer  2
                
                self.conv2 = nn.Conv2d(
                        in_channels = self.L1_OUT_CHANNEL,
                        out_channels = self.L2_OUT_CHANNEL,
                        kernel_size = self.L2_KERNEL_SIZE,
                        padding = (self.L2_KERNEL_SIZE - 1) // 2
                )
                self.batch_norm2 = nn.BatchNorm2d(self.L2_OUT_CHANNEL)
                
                # Layer  3
                self.conv3 = nn.Conv2d(
                        in_channels = self.L2_OUT_CHANNEL,
                        out_channels = self.L3_OUT_CHANNEL,
                        kernel_size = self.L3_KERNEL_SIZE,
                        padding = (self.L3_KERNEL_SIZE - 1) // 2
                )
                self.batch_norm3 = nn.BatchNorm2d(self.L3_OUT_CHANNEL)
                
                
                # Fully Connected (Dense) Layers
                self.dropout = nn.Dropout(self.DROPOUT_P)
                # After three MaxPools of 2x2, a 64x64 image becomes 8x8. 
                # So the flattened size is 128 channels * FINAL_DIM * FINAL_DIM
                
                self.fully_connected1 = nn.Linear(self.L3_OUT_CHANNEL * self.FINAL_DIM * self.FINAL_DIM, self.L1_NEURON)
                self.fully_connected2 = nn.Linear(self.L1_NEURON, num_classes)
                
        def forward(self, x: torch.Tensor):
                # L1
                x = self.pool(self.relu(self.batch_norm1(self.conv1(x))))
                
                # L2
                x = self.pool(self.relu(self.batch_norm2(self.conv2(x))))
                
                # L3
                x = self.pool(self.relu(self.batch_norm3(self.conv3(x))))
                
                x = x.view(-1, self.L3_OUT_CHANNEL * self.FINAL_DIM * self.FINAL_DIM)
                
                x = self.dropout(self.relu(self.fully_connected1(x)))
                x = self.fully_connected2(x)
                
                return x

In [6]:
model = CNN(num_classes=29).to(device)

In [7]:
def learning_process(model: nn.Module, criterion: nn.CrossEntropyLoss = nn.CrossEntropyLoss(), learning_rate = 0.001, epochs = 30):
        # criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
        
        train_losses, valid_losses = [], []
        train_accs, valid_accs = [], []
        
        best_val_loss = float('inf') # متغیری برای ذخیره بهترین Loss
        
        # Training
        for epoch in range(epochs):
                print(f"------------ {epoch = } Started")
                # Set model to training mode (enables Dropout and BatchNorm updates)
                model.train()
                
                running_loss, correct, total = 0.0, 0, 0
                for i, (images, labels) in enumerate(train_loader):
                        images, labels = images.to(device), labels.to(device)
                        # Clear old gradients
                        optimizer.zero_grad()
                        # Forward pass: predict labels
                        outputs = model(images)
                        # Calculate error
                        loss = criterion(outputs, labels)
                        # Backward pass: compute gradients
                        loss.backward()
                        # Update weights
                        optimizer.step()
                        
                        running_loss += loss.item()
                        _, predicted = torch.max(outputs.data, 1)
                        total += labels.size(0)
                        correct += (predicted == labels).sum().item()
                        
                train_losses.append(running_loss / len(train_loader))
                train_accs.append(100 * correct / total)
                
                print("----- Validation process Started")
                
                model.eval()
                valid_loss, correct, total = 0.0, 0, 0
                
                with torch.no_grad():
                        for images, labels in valid_loader:
                                images, labels = images.to(device), labels.to(device)
				# Clear old gradients
                                # optimizer.zero_grad()
				# Forward pass: predict labels
                                outputs = model(images)
				# Calculate error
                                loss = criterion(outputs, labels)
				# Backward pass: compute gradients
                                # loss.backward()
				# Update weights
                                # optimizer.step()
				
                                valid_loss += loss.item()
                                _, predicted = torch.max(outputs.data, 1)
                                total += labels.size(0)
                                correct += (predicted == labels).sum().item()
                valid_losses.append(valid_loss / len(valid_loader))
                valid_accs.append(100 * correct / total)
                
                print(f"Epoch {epoch+1}/{epochs} | "
                      f"Train Loss: {train_losses[-1]:.4f}, Acc: {train_accs[-1]:.2f}% | "
                      f"Val Loss: {valid_losses[-1]:.4f}, Acc: {valid_accs[-1]:.2f}%")
                
                if valid_losses[-1] < best_val_loss:
                        best_val_loss = valid_losses[-1]
                        torch.save(model.state_dict(), 'best_sign_language_model.pth')
                        print(f"Best model saved with Val Loss: {best_val_loss:.4f}")
        return train_losses, train_accs, valid_losses, valid_accs

In [8]:
train_losses, train_accs, valid_losses, valid_accs = learning_process(model = model)

------------ epoch = 0 Started
----- Validation process Started
Epoch 1/30 | Train Loss: 2.7033, Acc: 16.65% | Val Loss: 1.8403, Acc: 36.44%
Best model saved with Val Loss: 1.8403
------------ epoch = 1 Started
----- Validation process Started
Epoch 2/30 | Train Loss: 1.9352, Acc: 33.66% | Val Loss: 1.7297, Acc: 41.57%
Best model saved with Val Loss: 1.7297
------------ epoch = 2 Started
----- Validation process Started
Epoch 3/30 | Train Loss: 1.5949, Acc: 42.33% | Val Loss: 0.7696, Acc: 74.84%
Best model saved with Val Loss: 0.7696
------------ epoch = 3 Started
----- Validation process Started
Epoch 4/30 | Train Loss: 1.3542, Acc: 49.63% | Val Loss: 0.7648, Acc: 72.31%
Best model saved with Val Loss: 0.7648
------------ epoch = 4 Started
----- Validation process Started
Epoch 5/30 | Train Loss: 1.2142, Acc: 53.98% | Val Loss: 0.4396, Acc: 85.29%
Best model saved with Val Loss: 0.4396
------------ epoch = 5 Started
----- Validation process Started
Epoch 6/30 | Train Loss: 1.0753, Acc

In [9]:
torch.save(model.state_dict(), 'last_sign_language_model.pth')

print("Model saved successfully!")

Model saved successfully!


In [11]:
loaded_model = CNN(num_classes=29).to(device)
loaded_model.load_state_dict(torch.load('best_sign_language_model.pth'))
loaded_model.eval()

CNN(
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch_norm1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch_norm2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch_norm3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fully_connected1): Linear(in_features=8192, out_features=512, bias=True)
  (fully_connected2): Linear(in_features=512, out_features=29, bias=True)
)

In [12]:
def test_model(model):
	eval_model = model # or loaded_model

	# 2. Set the model to evaluation mode (Crucial: turns off Dropout and BatchNorm updates)
	eval_model.eval()

	# 3. Define the loss function (since it was previously trapped inside your learning_process function)
	criterion = nn.CrossEntropyLoss()

	test_loss = 0.0
	correct = 0
	total = 0

	# Lists to store all predictions and true labels for the Confusion Matrix
	all_preds = []
	all_true = []

	print("Starting evaluation on the Test Dataset...")

	# Turn off gradient calculation to save memory and speed up inference
	with torch.no_grad():
		for images, labels in test_loader:
			# Move data to the same device as the model (cuda or cpu)
			images, labels = images.to(device), labels.to(device)
			
			# Forward pass
			outputs = eval_model(images)
			loss = criterion(outputs, labels)
			
			# Track Loss and Accuracy
			test_loss += loss.item()
			_, predicted = torch.max(outputs.data, 1)
			
			total += labels.size(0)
			correct += (predicted == labels).sum().item()
			
			# Store predictions and true labels for the matrix
			all_preds.extend(predicted.cpu().numpy())
			all_true.extend(labels.cpu().numpy())

	# Calculate final averages
	avg_test_loss = test_loss / len(test_loader)
	test_accuracy = 100 * correct / total

	print("-" * 30)
	print(f"Final Test Loss: {avg_test_loss:.4f}")
	print(f"Final Test Accuracy: {test_accuracy:.2f}%")
	print("-" * 30)

	# 4. Generate and Plot the Confusion Matrix

	cm = confusion_matrix(all_true, all_preds)

	# Create custom colorscale with boundaries
	colors = ["#ffffff", "#d53737", "#3664a4", "#387d13"]
	boundaries = [0, 20, 300, 550, 999]

	# Normalize boundaries to 0-1 for colorscale
	min_val = min(boundaries)
	max_val = max(boundaries)
	normalized_boundaries = [(b - min_val) / (max_val - min_val) for b in boundaries]

	# Create colorscale
	colorscale = []
	for i in range(len(colors)):
		colorscale.append([normalized_boundaries[i], colors[i]])
		colorscale.append([normalized_boundaries[i+1], colors[i]])

	# Create heatmap
	fig = go.Figure(data=go.Heatmap(
		z=cm,  # your confusion matrix
		x=dataset_classes,
		y=dataset_classes,
		colorscale=colorscale,
		zmin=min(boundaries),
		zmax=max(boundaries),
		text=cm,
		texttemplate='%{text}',
		textfont={"size": 10},
		hovertemplate='True: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>'
	))

	# Update layout
	fig.update_layout(
		title='Confusion Matrix: Test Dataset',
		xaxis_title='Predicted Label',
		yaxis_title='True Label',
		width=800,
		height=700,
		xaxis={'tickangle': 45}
	)

	fig.show()

In [13]:
test_model(loaded_model)

Starting evaluation on the Test Dataset...
------------------------------
Final Test Loss: 0.0035
Final Test Accuracy: 99.93%
------------------------------


If your data is balanced but the model still fails, you need to force the model to pay more attention to the difficult class. You can do this by modifying your CrossEntropyLoss to weigh the errors differently.  The Logic: If the model gets a 'space' image wrong, we want the "fine" to be much higher than if it gets an 'A' image wrong.

In [ ]:
weights = [1.0] * 29
weights[dataset_classes.index('M')] = 2.0

class_weights = torch.tensor(weights) # Increase the weight for the difficult class
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

model_v2 = CNN(num_classes=29).to(device)
train_losses, train_accs, valid_losses, valid_accs = learning_process(model = model_v2, criterion = criterion)
torch.save(model.state_dict(), 'sign_language_modelv2.pth')
test_model(model = model_v2)
